In [ ]:
!pip install -q datasets librosa tensorflow scikit-learn

In [ ]:
from datasets import load_dataset

# Load the DFADD dataset
ds = load_dataset(
    "SpeechAntiSpoofingBenchmarks/DFADD",
    split="test"
)

print("Dataset loaded successfully!")
print(ds)
print("Available columns:", ds.column_names)

Dataset loaded successfully!
Dataset({
    features: ['path', 'audio', 'label', 'notes'],
    num_rows: 3755
})
Available columns: ['path', 'audio', 'label', 'notes']


In [ ]:
import numpy as np
import librosa
import tensorflow as tf

X = []
y = []

for row in ds:
    audio = row["audio"]
    signal = np.asarray(audio["array"], dtype=np.float32)
    sr = audio["sampling_rate"]

    # Create mel-spectrogram
    mel = librosa.feature.melspectrogram(
        y=signal,
        sr=sr,
        n_mels=128
    )

    mel = librosa.power_to_db(mel, ref=np.max)

    # Normalize
    mel = (mel - mel.min()) / (mel.max() - mel.min() + 1e-8)

    # Resize to 128 x 128
    mel = tf.image.resize(
        mel[..., np.newaxis],
        (128, 128)
    ).numpy()

    # Convert grayscale to RGB for VGG16
    mel = np.repeat(mel, 3, axis=-1) * 255.0

    X.append(mel)
    y.append(row["label"])

X = np.asarray(X, dtype=np.float32)
y = np.asarray(y, dtype=np.int32)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1510, 128, 128, 3)
y shape: (1510,)


In [ ]:
import numpy as np
import librosa
import tensorflow as tf
from datasets import load_dataset

# Load dataset
ds = load_dataset(
    "SpeechAntiSpoofingBenchmarks/DFADD",
    split="test"
)

# Select real and spoof audio
real = [i for i, row in enumerate(ds) if row["label"] == 0]
fake = [i for i, row in enumerate(ds) if row["label"] == 1]

rng = np.random.default_rng(42)

n = min(len(real), len(fake), 755)

real = rng.choice(real, n, replace=False)
fake = rng.choice(fake, n, replace=False)

indices = np.concatenate([real, fake])
rng.shuffle(indices)

ds = ds.select(indices.tolist())

print("Total audio clips:", len(ds))

Total audio clips: 1510


In [ ]:
IMG_SIZE = 128

X = []
y = []

for i, row in enumerate(ds):
    audio = row["audio"]

    signal = audio["array"]
    sr = audio["sampling_rate"]

    # Convert audio into a mel-spectrogram
    mel = librosa.feature.melspectrogram(
        y=signal.astype(np.float32),
        sr=sr,
        n_mels=128
    )

    mel = librosa.power_to_db(mel, ref=np.max)

    # Normalize
    mel = (mel - mel.min()) / (
        mel.max() - mel.min() + 1e-8
    )

    # Resize image
    mel = tf.image.resize(
        mel[..., np.newaxis],
        (IMG_SIZE, IMG_SIZE)
    ).numpy()

    # Convert grayscale to RGB for VGG16
    image = np.repeat(mel, 3, axis=-1) * 255

    X.append(image)
    y.append(row["label"])

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int32)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Dataset preprocessing completed!")

X shape: (1510, 128, 128, 3)
y shape: (1510,)
Dataset preprocessing completed!


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 1208
Testing samples: 302


In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam

# Load the VGG16 architecture
base_model = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(128, 128, 3)
)

# Freeze pretrained VGG16 layers
base_model.trainable = False

# Add classification layers for VoiceGuard
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.5)(x)

# Binary classification: real vs fake
predictions = Dense(1, activation="sigmoid")(x)

# Create the complete model
model = Model(inputs=base_model.input, outputs=predictions)

# Compile the model
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 128, 128, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 128, 128, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 64, 64, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 64, 64, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 32, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 32, 32, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 32, 32, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 32, 32, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 16, 16, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 16, 16, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 16, 16, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 16, 16, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 8, 8, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 8, 8, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 8, 8, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 8, 8, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 4, 4, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        65,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,780,481 (56.38 MB)

 Trainable params: 65,793 (257.00 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=16,
    validation_split=0.2,
    callbacks=[early_stopping]
)

Epoch 1/10
61/61 ━━━━━━━━━━━━━━━━━━━━ 206s 3s/step - accuracy: 0.5466 - loss: 2.2475 - val_accuracy: 0.6860 - val_loss: 0.5909
Epoch 2/10
61/61 ━━━━━━━━━━━━━━━━━━━━ 254s 3s/step - accuracy: 0.6636 - loss: 1.4322 - val_accuracy: 0.8223 - val_loss: 0.3837
Epoch 3/10
61/61 ━━━━━━━━━━━━━━━━━━━━ 199s 3s/step - accuracy: 0.7340 - loss: 0.9106 - val_accuracy: 0.8512 - val_loss: 0.3151
Epoch 4/10
61/61 ━━━━━━━━━━━━━━━━━━━━ 210s 3s/step - accuracy: 0.7712 - loss: 0.6900 - val_accuracy: 0.8760 - val_loss: 0.2678
Epoch 5/10
61/61 ━━━━━━━━━━━━━━━━━━━━ 199s 3s/step - accuracy: 0.8137 - loss: 0.5606 - val_accuracy: 0.9050 - val_loss: 0.2672
Epoch 6/10
61/61 ━━━━━━━━━━━━━━━━━━━━ 201s 3s/step - accuracy: 0.8375 - loss: 0.4627 - val_accuracy: 0.9298 - val_loss: 0.2050
Epoch 7/10
61/61 ━━━━━━━━━━━━━━━━━━━━ 200s 3s/step - accuracy: 0.8685 - loss: 0.3716 - val_accuracy: 0.9339 - val_loss: 0.1880
Epoch 8/10
61/61 ━━━━━━━━━━━━━━━━━━━━ 197s 3s/step - accuracy: 0.8530 - loss: 0.3738 - val_accuracy: 0.9380 - v

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

probabilities = model.predict(X_test).ravel()
predictions = (probabilities >= 0.5).astype(int)

print("Accuracy:", accuracy_score(y_test, predictions))
print("Precision:", precision_score(y_test, predictions, zero_division=0))
print("Recall:", recall_score(y_test, predictions, zero_division=0))
print("F1 Score:", f1_score(y_test, predictions, zero_division=0))

10/10 ━━━━━━━━━━━━━━━━━━━━ 50s 5s/step
Accuracy: 0.956953642384106
Precision: 0.9423076923076923
Recall: 0.9735099337748344
F1 Score: 0.9576547231270358
